In [ ]:
# ===== REFINED TOP-3 CoV on ms640 @ budget 100,000 — CONFIG ==============
# ===== edit ONLY this cell ===============================================
#
# WHAT THIS RUNS. The same two-stage pipeline as cov_top3_ms640_abel.ipynb /
# _len.ipynb, with a refined ranking. For each of the 640 ms640 presentations
# Stage A enumerates every valid change-of-variables candidate and picks 3;
# Stage B searches all three in rank order at BUDGET nodes each. What changed is
# how the 3 are picked:
#
#   1. rank by the arm's key            (abel | abel+total length | total length)
#   2. then by S, the smaller mean block <- the tie-break, "_s" rules
#   3. drop any candidate that is an earlier pick with x and y renamed
#   4. pull deeper into the same ranking to refill the freed slot
#
# Step 3 is the point. The 8 signed permutations of {x, y} are automorphisms, so
# a solution exists for one iff it exists for all eight and the minimal AC path
# length is identical — a slot holding a relabel of an earlier pick re-searches
# a start this arm has already searched. The shipped abel manifest spends 723
# such slots over 500 of the 640 presentations; the shipped len manifest 424
# over 343. Every rule below spends none.
#
# WHAT THIS IS EXPECTED TO SHOW, so a null result is not read as a bug. On the
# frozen subset-60 sweeps (where every candidate of every row was searched, so a
# promoted pick can be priced without new search) the dedup is *paired-identical*
# at budget 1,000 and 10,000 on all three keys — 0 wins, 0 losses. It is hygiene:
# k should mean k distinct searches. Its one measured gain there is (total) at
# 10,000, top-3 49 -> 50. MK is the genuinely open question: it helps (abel) at
# 1,000 and HURTS it at 10,000, and S hurts that arm at both — but (abel) is not
# an arm anyone runs. On (abel, total) S and MK are indistinguishable. Subset-60 is 60 rows and those margins are 1-2
# rows wide, which the repo's own control-with-no-dynamic-range and gap-metric
# lessons say is not yet a property of the key. See COV_RELABEL_B1K.md.
#
# WHICH ARMS ARE WORTH THE NODES — read before queueing a session.
#
#   abel_rd        DO NOT RUN. Provably a no-op, settled today at zero search.
#                  The dedup keeps the first-ranked member of each relabel class,
#                  so it never changes rank 1 (verified: rank 1 identical on
#                  640/640 vs shipped abel), and shipped abel's rank 1 ALREADY
#                  SOLVES 640/640 at 100,000. Its solve count and its 458,688-node
#                  rank-1 bill are identical to the shipped arm by construction.
#   len_rd         RUN FIRST. The only arm with real dynamic range: len's rank 1
#                  fails on 7 presentations (425, 435, 573, 599, 601, 634, 635)
#                  and the dedup rewrites the top 3 on 5 of them.
#   abel_len_rd    the control for the third term. Run paired with the next one.
#   abel_len_rd_s  THE RULE — abel -> length -> S. The pair is powered on a cost
#                  comparison, NOT on solves, which are pinned at the 640/640
#                  ceiling for every abel-first arm at this budget.
#   len_rd_s       S on the length arm. Subset-60 likes this one: vs len_rd it is
#                  -171 nodes at budget 1,000 and -6,395 with top-3 +2 at 10,000,
#                  where MK manages +1,000 and -2,125 / +1.
#   *_rd_mk        MK instead of S, kept only as a comparison. On (abel, total) the
#                  two are indistinguishable (8 nodes apart on ~294k); on the
#                  length arm S is clearly better. Not the recommendation.
#
# So this run is a powered test of COST and of 17 rank-1 changes. It cannot be a
# powered test of the solve rate on an abel arm, because that metric has no
# headroom left on ms640.
#
# ALL THREE RANKS ALWAYS RUN, including ranks below one that already solved, so
# every rank is measured on the same 640 presentations; under early exit ranks
# 2-3 would exist only where rank 1 failed and their means would describe a
# harder, self-selected subset. The deployed early-exit cost is not lost —
# summarize() recovers it exactly as first_solve_nodes. A solve does NOT mark a
# presentation done, or a restart would skip precisely the easy ones.
#
# ONE ARM PER SESSION. RULE is the only knob that differs between arms; each
# writes its own jsonl (the rule is in the filename) and none touches another's.
# Open a second Colab session on this same notebook to run a second arm.
#
# RESTART CONTRACT. Runtime -> Restart, then Run All, continues this run: SETUP
# resets the repo to the latest push, purges the stale experiments.* modules,
# and seeds the local jsonl back from Drive; RUN resumes from it. Mid-run
# hotfixes must be pushed as .py files — a pushed .ipynb does NOT reach an
# already-open Colab notebook.

REPO_URL = "https://github.com/Avi161/ACSolverX.git"
BRANCH   = "research/w5/stable-ac-escape"   # must match the actual git branch
REPO_DIR = "ACSolverX"
CLONE       = True
UPDATE_REPO = True           # git reset --hard so a RESTART pulls latest push

MOUNT_DRIVE = True           # mirror the results jsonl to Drive every few
                             # minutes + at the end, and seed it BACK on a fresh
                             # VM so resume continues where the last session
                             # stopped. The runner always writes locally
                             # (appending onto the Drive FUSE mount is unsafe).
DRIVE_DIR   = "/content/drive/MyDrive/acsolverx_results/cov_top3"

# --- experiment knobs ------------------------------------------------------
# RULE is the only knob that differs between arms. The five, and why each exists:
#
#   "abel_rd"         (abel)              the SHIPPED abel arm + dedup, and
#                                         nothing else -> the one-variable test
#                                         of what the dedup alone is worth
#   "len_rd"          (total)             the SHIPPED len arm + dedup, likewise
#   "abel_len_rd"     (abel, total)       the b1k-validated ranking + dedup
#   "abel_len_rd_mk"  (abel, total, MK)   the above + the tie-break feature
#                                         <- the recommended arm
#   "len_rd_mk"       (total, MK)         the length arm + the tie-break feature
#
# Run abel_rd / len_rd if you want the dedup priced against the two frozen
# shipped runs; run abel_len_rd + abel_len_rd_mk as a pair to isolate MK.
RULE        = "abel_len_rd_s"
BUDGET      = 100_000        # PER SEARCH; a presentation costs <= K * this
K           = 3              # ranks per presentation (the manifests are built at 3)
HIGH_SPEEDUP = True          # compact fast solver (~2.9x); result-neutral —
                             # a solved fast search is re-solved by the normal
                             # solver for its path, so every written row is
                             # identical to a slow-mode row and the files resume
                             # across the two modes
CHUNKS      = 1              # this arm is one session; raise it (with
CHUNK_INDEX = None           # CHUNK_INDEX = 1..CHUNKS) only to split ONE arm
                             # across more machines, then run the MERGE cell


In [ ]:
# ==================== SETUP (clone / pull / Drive) ========================
import os, sys, subprocess

def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-2000:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-2000:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB)

if IN_COLAB:
    BASE = "/content"
    os.chdir(BASE)                       # anchor so re-runs never nest the clone
    if not os.path.isdir(REPO_DIR):
        if CLONE:
            sh(f"git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}")
    elif UPDATE_REPO:
        sh(f"cd {REPO_DIR} && git fetch --depth 1 origin {BRANCH} && git reset --hard FETCH_HEAD")
    sh(f"cd {REPO_DIR} && git log -1 --oneline")
    sh("pip -q install numba numpy pyyaml")
    REPO_ROOT = os.path.join(BASE, REPO_DIR)
else:
    # local: walk up from cwd to the repo root (dir holding experiments/ + data/)
    REPO_ROOT = os.getcwd()
    while REPO_ROOT != "/" and not (
        os.path.isdir(os.path.join(REPO_ROOT, "experiments"))
        and os.path.isdir(os.path.join(REPO_ROOT, "data"))
    ):
        REPO_ROOT = os.path.dirname(REPO_ROOT)

os.chdir(REPO_ROOT)                      # relative paths + "import experiments…"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("repo root:", REPO_ROOT)

# a `git reset --hard` rewrites .py files but sys.modules keeps the OLD module
# objects -- drop them so RUN imports what SETUP just fetched (pull != reload)
import importlib
for _m in [m for m in sys.modules if m == "experiments" or m.startswith("experiments.")]:
    del sys.modules[_m]
importlib.invalidate_caches()

# --- Drive: mount + seed-back (fresh VM -> local resume state) -------------
import glob, shutil
LOCAL_OUT = os.path.join(REPO_ROOT, "results", "stable_ac", "cov", "cov_top3")
os.makedirs(LOCAL_OUT, exist_ok=True)
if IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    for src in glob.glob(os.path.join(DRIVE_DIR, "*.jsonl")):
        dst = os.path.join(LOCAL_OUT, os.path.basename(src))
        # the bigger file wins: local mid-run state beats a stale mirror, and on
        # a fresh VM the mirror beats the (absent/empty) local file
        if not os.path.exists(dst) or os.path.getsize(dst) < os.path.getsize(src):
            shutil.copyfile(src, dst)
            print("seeded from Drive:", os.path.basename(src))


In [ ]:
# ==================== RUN =================================================
# Production budgets run HERE, never on the dev machine: the repo caps any
# locally-launched search at 1,000 nodes, and the runner enforces that cap
# unless this flag is set.
os.environ["ACSOLVERX_ALLOW_BIG"] = "1"

from experiments.stable_ac.cov.run import cov_top3_relabel as rd
from experiments.stable_ac.cov.run import cov_top3_run as R

# Stage A is committed (1,920 picks over 640 presentations per rule) — rebuild
# only if a shallow clone somehow lacks it. It explores ZERO nodes and takes
# ~3 s. The manifest path is DERIVED from RULE, never passed beside it: a stale
# path plus another rule's name is a wrong experiment wearing a plausible
# filename.
MANIFEST = rd.manifest_path(RULE)
if not os.path.exists(os.path.join(REPO_ROOT, MANIFEST)):
    print("manifest missing — rebuilding (no search)")
    rd.build(rule=RULE, out_path=MANIFEST)

# THE GATE, before a single node is spent. A deduped manifest path is still
# writable by an undeduped build, and the rule name alone cannot detect that —
# so preflight re-derives every pick's relabel class from its own (r1, r2)
# rather than trusting the stored tag, and refuses a file that has a repeat, was
# built for another rule, or is missing.
from experiments.stable_ac.cov.run import cov_top3_relabel_run as rdrun
_pf = rdrun.preflight(RULE)
print(f"[preflight] {_pf['path']}: no relabel repeats over {_pf['n_pres']} presentations")

groups = rd.load_manifest(MANIFEST, rule=RULE)
print(f"manifest [{RULE}]: {len(groups)} presentations x <= {K} ranks")

# mirror local jsonls -> Drive every 3 min (and once at the end). Whole-file
# copies of an append-only jsonl: a torn tail line is repaired on resume. The
# thread never prints (a background thread must not).
import threading
def _sync_to_drive():
    if not (IN_COLAB and MOUNT_DRIVE):
        return
    for src in glob.glob(os.path.join(LOCAL_OUT, "*.jsonl")):
        dst = os.path.join(DRIVE_DIR, os.path.basename(src))
        # size-monotonic: an append-only jsonl only ever grows, so never
        # overwrite a bigger Drive copy with a smaller local one (a session
        # holding a stale seeded copy of another arm must not clobber it)
        if not os.path.exists(dst) or os.path.getsize(dst) < os.path.getsize(src):
            tmp = dst + ".tmp"
            shutil.copyfile(src, tmp)
            os.replace(tmp, dst)
def _mirror_loop():
    while not _mirror_stop.wait(180):
        try: _sync_to_drive()
        except Exception: pass                 # transient Drive hiccup: next tick
_mirror_stop = threading.Event()
threading.Thread(target=_mirror_loop, daemon=True).start()

# `registered` splices RULE into the shipped whitelist for the duration of the
# run and takes it back out afterwards, including on error. It is NOT permanent:
# cov_top3_manifest.RULES is read at module scope elsewhere (five tests
# parametrize on it at collection time, and its no-argument CLI builds its
# build-everything list from it, which would overwrite these manifests with
# undeduped ones). run / summarize / merge_chunks all validate through
# load_config, so all of them belong inside the block.
try:
    with rd.registered(RULE):
        out_path = R.run(rule=RULE, budget=BUDGET, k=K, chunks=CHUNKS,
                         chunk_index=CHUNK_INDEX, high_speedup=HIGH_SPEEDUP,
                         manifest=MANIFEST)
        # This arm's score. The plain-greedy controls are read by TRUNCATING the
        # frozen 1,000,000-node ms640 baseline (zero new search) at BUDGET and at
        # K x BUDGET, and the paired nodes/path comparison runs over the
        # presentations BOTH arms solved. Then the gate: every search that
        # overlaps the frozen 10,000-node subset-60 sweep must reproduce it node
        # for node.
        R.summarize(out_path, rule=RULE, budget=BUDGET, k=K, chunks=CHUNKS,
                    chunk_index=CHUNK_INDEX, manifest=MANIFEST)
finally:
    _mirror_stop.set()
    _sync_to_drive()                           # final sync, incl. the last rows
    if IN_COLAB and MOUNT_DRIVE:
        print("mirrored to", DRIVE_DIR)


In [ ]:
# ============ MERGE / COMPARE (optional, after the other arms finish) ======
# Separate cell because each half needs a file this session did not write: run
# it only once the run(s) it names have printed their "done" line and mirrored
# to Drive. It re-seeds from Drive first, so run it in whichever session you like.
for src in glob.glob(os.path.join(DRIVE_DIR, "*.jsonl")) if (IN_COLAB and MOUNT_DRIVE) else []:
    dst = os.path.join(LOCAL_OUT, os.path.basename(src))
    if not os.path.exists(dst) or os.path.getsize(dst) < os.path.getsize(src):
        shutil.copyfile(src, dst)

# (a) only if you split THIS arm across several machines (CHUNKS > 1)
if CHUNKS > 1:
    with rd.registered(RULE):
        out_path = R.merge_chunks(rule=RULE, budget=BUDGET, k=K, chunks=CHUNKS,
                                  manifest=MANIFEST)
        R.summarize(out_path, rule=RULE, budget=BUDGET, k=K, chunks=CHUNKS,
                    manifest=MANIFEST)

# (b) head-to-head against whichever other arms have finished. Scored on the
# presentations both arms searched — with sessions finishing at different times,
# an intersection is the only denominator both have earned.
#
# Read the COST columns, not just the solve count. Both shipped arms already
# reach 640/640 and 638/640 at this budget, so a refined arm has almost no room
# to add solves and will look inert on that column by construction; what it can
# move is nodes, and rank 1. The pairs worth naming:
#
#   abel  vs abel_rd            what the dedup alone is worth
#   len   vs len_rd             the same, on the length arm
#   abel_len_rd vs abel_len_rd_s    what S alone is worth, powered at last
def _arm(rule):
    hits = [p for p in glob.glob(os.path.join(LOCAL_OUT, f"{rule}top{K}_{BUDGET}_*.jsonl"))
            if not R._CHUNK_MARK.search(os.path.basename(p))]
    return max(hits, key=lambda p: sum(1 for _ in open(p))) if hits else None

PAIRS = [("abel", "abel_rd"), ("len", "len_rd"),
         ("abel_len_rd", "abel_len_rd_s"), ("len_rd", "len_rd_s")]
for a_rule, b_rule in PAIRS:
    a, b = _arm(a_rule), _arm(b_rule)
    if a and b:
        print(f"\n===== {a_rule}  vs  {b_rule} =====")
        R.compare_rules(a, b, k=K, label_a=a_rule, label_b=b_rule)
    else:
        absent = [r for r, p in ((a_rule, a), (b_rule, b)) if not p]
        print(f"skip {a_rule} vs {b_rule}: no results file yet for {absent}")
